# exp_6 · 01 simulator rechecks (IDSSE only)
1. **Recheck (binding).** S(c = 0) tokens equal exp_2's cached `target_like` tokens, and its rows equal exp_3's
   stored `exp3_ref_ens3` `target_like` rows on provider and `L_ctrl`.
2. **Recheck (report).** exp_3's S2 coverage view regenerated in memory reproduces exp_3's S2 rows and deltas.
3. **Context recheck (report).** `exp5_touch_ens3` through the same path vs exp_5's stored rows.
The VOI grid runs in 03 (PREREG §0 deviation: it needs σ̂, f̂ and the run lengths from 02).

In [1]:
from pathlib import Path
import os, sys, json, time, subprocess
from datetime import datetime, timezone
ROOT = Path.cwd()
if not (ROOT / 'exp6_common.py').exists():
    raise RuntimeError('Run with working directory exp_jeremy/exp_6')
sys.path.insert(0, str(ROOT))
sys.dont_write_bytecode = True
import exp6_common as C
import exp1_common as C1
import numpy as np
import pandas as pd
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)
def now():
    return datetime.now(timezone.utc).isoformat(timespec='seconds')
def make_log(name):
    path = C.LOG_DIR / f'{name}_progress.log'
    path.parent.mkdir(parents=True, exist_ok=True)
    def log(message, **_):
        print(message, flush=True)
        with path.open('a') as handle:
            handle.write(f'{now()} {message}\n')
    return log
protocol = C.verify_prereg()
print('PREREG verified', protocol['prereg_sha256'][:16], '| frozen', protocol['frozen_at_utc'])

import exp6_sim as S
log = make_log('01')

PREREG verified dca69fd7188c7ef8 | frozen 2026-11-10T14:57:58+00:00


In [2]:
rows = S.evaluate([S.BASE, S.Setting(S.S2_RECHECK, 0.12, 0.0, 0.0)], np.array([1]), 'reference', log)
touch_rows = S.evaluate([S.BASE], np.array([1]), 'touch', log)
rechecks = {'checked_at_utc': now(), 'tokens': S.recheck_tokens(), 'target_like_rows': S.recheck_base(rows),
            's2_coverage': S.recheck_s2(rows), 'touch_target_like_rows': S.recheck_base(touch_rows, 'touch')}
C1.write_json(C.RESULTS / 'sim_rechecks.json', rechecks)
pd.DataFrame(rows + touch_rows).to_csv(C.RESULTS / 'sim_rows_01.csv', index=False)
for key, value in rechecks.items():
    print(key, value)
if not (rechecks['tokens']['all_equal'] and rechecks['target_like_rows']['rows_equal']):
    raise RuntimeError('Binding recheck failed: S(0) does not reproduce exp_3 target_like')

reference target_like J03WMX 3.6s


reference exp3_s2_coverage J03WMX 2.8s


reference target_like J03WN1 1.7s


reference exp3_s2_coverage J03WN1 1.9s


reference target_like J03WOH 1.7s


reference exp3_s2_coverage J03WOH 2.0s


... (12 outputs left out)

touch target_like J03WPY 1.9s


touch target_like J03WQQ 1.9s


touch target_like J03WR9 2.0s


checked_at_utc 2026-11-10T14:59:26+00:00
tokens {'per_match': {'J03WMX': True, 'J03WN1': True, 'J03WOH': True, 'J03WOY': True, 'J03WPY': True, 'J03WQQ': True, 'J03WR9': True}, 'all_equal': True}
target_like_rows {'rows': 14, 'max_abs_diff': 0.0, 'rows_equal': True, 'max_cached_probability_diff': 0.0}
s2_coverage {'rows': 14, 'max_abs_diff': 1.1102230246251565e-16, 'rows_equal': False, 'delta_provider': {'mine': 0.00951008389840023, 'stored': 0.00951008389840023}, 'delta_ctrl': {'mine': 0.006918298233357466, 'stored': 0.006918298233357466}}
touch_target_like_rows {'rows': 14, 'max_abs_diff': 0.0, 'rows_equal': True}
